# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 24_25_optimizer_workflow_with_cost_and_ILP  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-02-27
### Properties script
**Goal:** to generate a cost matrix for the geometry with use of the timber datasets, then using ILP to find the best matches   
**Inputs:**
*   CSV timber dataset
*   Digital geometry

**Outputs:**
*   Best match for each element in a structure

# IMPORTING

## Dataset

In [ ]:
import config
import pandas as pd

# Laad de geëxporteerde dataset in je nieuwe omgeving
# Voeg sep=';' toe aan je inlaad-functie!
#df_input_stock = pd.read_csv(config.RAW_DATA_PATH + 'complete_timber.csv', sep=';')
df_input_stock = pd.read_csv(config.RAW_DATA_PATH / 'complete_timber_small_260310.csv', sep=';', encoding='latin1')

# Print de kolommen om zeker te weten dat ze goed gesplitst zijn
print("Gevonden kolommen:", df_input_stock.columns.tolist())

# Controleer of alle data, inclusief de RS_0000 ID's en binaire states, goed is overgekomen
print("\nDataset succesvol ingeladen. Hier zijn de eerste elementen:\n")
print(f"Dataset bevat {df_input_stock.shape[0]} elementen\n")
print(df_input_stock.head())

## Search space

De search space wordt vanuit het geometrie script geimporteerd, dan wordt een willekeurige samenstelling gekozen als beginpunt van de optimalisatie.


In [ ]:
import json

json_path = 'search_space.json'
# Lees het "contract" in voor je optimizer
with open(json_path, 'r') as f:
    optimizer_search_space = json.load(f)

print(f"✅ Search Space ingeladen! De optimizer heeft {len(optimizer_search_space)} parameters om aan te draaien.")
# Nu kun je deze variabele direct aan je Optuna, PyTorch of Genetisch Algoritme voeren!

Dit script is bedoeld voor je Colab-omgeving. Het leest de search_space.json in, stelt de parameters dynamisch in op basis van jouw regels, en gebruikt een "dummy" voorspelling (waar later je echte Neurale Netwerk komt) om het beste ontwerp te vinden.

# GEOMETRY

## Random geometry

In [ ]:
import json
import random
import c11_params

# ==========================================
# 1. RANDOM PARAMETERS GENEREREN (De "DNA" string)
# ==========================================
def get_random_design(json_path):
    """Leest de search space en trekt voor elke knop een willekeurige waarde."""
    with open(json_path, 'r') as f:
        search_space = json.load(f)

    random_params = {}

    for var_name, rules in search_space.items():
        if rules['type'] == 'continuous':
            # Kies een willekeurig kommagetal (bijv. voor u en v)
            random_params[var_name] = random.uniform(rules['min'], rules['max'])

        elif rules['type'] == 'discrete':
            # Kies exact één van de toegestane stapjes (bijv. voor shift_x)
            random_params[var_name] = random.choice(rules['options'])

    return random_params

# --- TEST DE FUNCTIE ---
print("Stap 1: Willekeurig DNA genereren...")
mijn_random_ontwerp = get_random_design(json_path)

print("\nSucces! Dit is het DNA van ons proef-ontwerp:")
for sleutel, waarde in mijn_random_ontwerp.items():
    print(f" - {sleutel}: \t{waarde:.3f}")

# Opzet

In [ ]:
import pandas as pd
from geometry import generate_sample_vertices
from reconstruction import reconstruct_edges

# --- UITVOEREN VAN STAP 2 ---
print("Stap 2: DNA vertalen naar 3D Geometrie...")

# We gebruiken 'mijn_random_ontwerp' uit de VORIGE cel!
vertices_list = generate_sample_vertices(sample_id=0, params=mijn_random_ontwerp)
df_vertices = pd.DataFrame(vertices_list)
df_edges = reconstruct_edges(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)

print(f"✅ Succes! Geometrie opgebouwd met {len(df_vertices)} knooppunten en {len(df_edges)} staven.")
print("\nVoorbeeld van de gegenereerde Vertices (eerste 5):")
print(df_vertices.head())

## Feature extraction

In [ ]:
import pandas as pd
from reconstruction import extract_beam_properties

# --- UITVOEREN VAN STAP 3 ---
print("Stap 3: Staaflengtes en profielen berekenen...")

# We gebruiken de tabellen uit de VORIGE cel!
df_slots = extract_beam_properties(df_vertices, df_edges)

print(f"✅ Succes! We hebben de eigenschappen van {len(df_slots)} balken berekend.")
print("\nDit is de input (Slots) voor je Cost Matrix (eerste 10 balken):")
print(df_slots.head(10))

# COST MATRIX

Omdat de pseudo-LCA berekening (uit de vorige stap) de impact van de gehele balk al berekent, heb je nu een methodologische keuze:

Optie A (De dubbele boete): Je telt de LCA-score én de penalty's bij elkaar op. Hiermee straf je afval extra zwaar af ten opzichte van het puur inbrengen van materiaal. Dit dwingt het algoritme agressief naar optimaal materiaalgebruik.

Optie B (De uitsplitsing): Je berekent de LCA-score alleen over het nuttige deel, en gebruikt je nieuwe formules om het verlies in kaart te brengen.

In [ ]:
# ==========================================
# CEL 1: INPUTS EN ALGEMENE PARAMETERS
# ==========================================
import pandas as pd
import numpy as np
import c11_params

print("✅ Parameters succesvol geladen.")

## Controle

In [ ]:
from cost_calculation import calculate_pseudo_lca_stock

# 1. Voer de functie uit op je reeds ingeladen DataFrame
df_stock_with_lca = calculate_pseudo_lca_stock(df_input_stock)

# 2. Bekijk de resultaten van de berekening
print("\nPreview van de berekende E_cost per element:")
display(df_stock_with_lca.sample(10))

## Building of cost matrix

In [ ]:
# Print even ter controle hoeveel balken erin zitten
print(f"Hout-inventaris succesvol ingeladen: {len(df_input_stock)} balken beschikbaar.")

In [ ]:
# --- Controle van de 
# Bepaal automatisch twee balken om te onderzoeken (1x RS en 1x NS)
rs_balken = df_input_stock[df_input_stock['Member_ID'].str.contains('RS')]['Member_ID'].tolist()
ns_balken = df_input_stock[df_input_stock['Member_ID'].str.contains('NS')]['Member_ID'].tolist()

balk_onderzoek = []
if rs_balken: balk_onderzoek.append(rs_balken[0]) # Pak de eerste Reclaimed balk
if ns_balken: balk_onderzoek.append(ns_balken[0]) # Pak de eerste New balk

print(f"🔍 Diepte-analyse gestart voor: {balk_onderzoek}")

In [ ]:
import numpy as np
import pandas as pd
import config
from cost_calculation import build_cost_matrix

# Bepaal automatisch twee balken om te onderzoeken (1x RS en 1x NS)
rs_balken = df_input_stock[df_input_stock['Member_ID'].str.contains('RS')]['Member_ID'].tolist()
ns_balken = df_input_stock[df_input_stock['Member_ID'].str.contains('NS')]['Member_ID'].tolist()

balk_onderzoek = []
if rs_balken: balk_onderzoek.append(rs_balken[0]) # Pak de eerste Reclaimed balk
if ns_balken: balk_onderzoek.append(ns_balken[0]) # Pak de eerste New balk

print(f"🔍 Diepte-analyse gestart voor: {balk_onderzoek}")

# Bouw de matrix en haal het logboek (df_logs) op
cost_matrix, verrijkte_stock, df_logs = build_cost_matrix(df_slots, df_input_stock, target_stock_ids=balk_onderzoek)

# 3. Presentatie van het resultaat
# Maak er een DataFrame van, puur om het netjes te kunnen printen.
df_cost_matrix_display = pd.DataFrame(
    cost_matrix,
    index=[f"{row['edge_id']} ({row['Length_Req']:.0f}mm)" for _, row in df_slots.iterrows()],
    columns=verrijkte_stock['Member_ID'].tolist()
)

df_cost_matrix_display.to_csv(config.EXPORT_PATH /'final_cost_matrix.csv', index=True)

# --- PRESENTEER DE DIEPTE-ANALYSE ---
print("\n" + "="*80)
print("🔬 DIEPTE-ANALYSE: ONLEDING VAN DE CO2 PENALTY")
print("="*80)

# Sorteer het logboek netjes op Stock_ID
if not df_logs.empty:
    df_logs = df_logs.sort_values(by=['Stock_ID', 'Slot_ID']).reset_index(drop=True)
    # Print als nette tabel, verwijder de NaN voor overzichtelijkheid
    print(df_logs.fillna('-').to_string(index=False))
else:
    print("Geen logboek data gevonden.")

print("="*80)

print("\nPreview Cost Matrix (eerste 8 staven, eerste 5 inventaris-balken):")
print("(Let op: 'inf' betekent dat de inventaris-balk te klein is en wordt uitgesloten)\n")
print("=" *80)
print(df_cost_matrix_display.iloc[:8, :5].round(2))

# MATCHING ALGORITHM / ILP

## Script

In [ ]:
import pulp
import numpy as np
import pandas as pd

# ==========================================
# CEL 4: TOEPASSING VAN HET OPTIMALISATIE ALGORITME (MILP)
# ==========================================
print("Start MILP Optimizer voor definitieve toewijzing...")

# 1. DATA KOPPELEN VANUIT VORIGE SCRIPTS
# Haal de namen op uit de DataFrames van de vorige cellen
stock_items = verrijkte_stock['Member_ID'].tolist()
construction_slots = df_slots['edge_id'].tolist()

# Dynamische categorisatie (kijkt naar 'NS' en 'RS' in de Member_ID)
new_items = [item for item in stock_items if 'NS' in item]
reclaimed_items = [item for item in stock_items if 'RS' in item]

print(f"📊 Inventaris: {len(reclaimed_items)} Reclaimed, {len(new_items)} New elementen.")
print(f"📐 Constructie vereist: {len(construction_slots)} staven.")

# 2. FILTEREN VAN DE COST MATRIX (Sparse Matrix Maken)
# We maken alleen combinaties aan die fysiek passen (waar de cost NIET oneindig is)
valid_matches = []
costs = {}

for i, slot_id in enumerate(construction_slots):
    for j, stock_id in enumerate(stock_items):
        cost = cost_matrix[i, j]
        # Als de cost niet 'inf' is, is het een geldige match!
        if cost != np.inf:
            valid_matches.append((stock_id, slot_id))
            costs[(stock_id, slot_id)] = cost

print(f"⚡ Aantal geldige combinaties voor de solver gereduceerd tot: {len(valid_matches)}")

# 3. HET MODEL OPZETTEN
prob = pulp.LpProblem("Reclaimed_Timber_Matching", pulp.LpMinimize)

# 4. DECISION VARIABLES
# We maken uitsluitend 'knoppen' aan voor de combinaties die kunnen passen
x = pulp.LpVariable.dicts("Match", valid_matches, 0, 1, pulp.LpBinary)

# 5. OBJECTIVE FUNCTION (Doel: Minimaliseer de totale CO2 penalty)
prob += pulp.lpSum([x[match] * costs[match] for match in valid_matches])

# 6. CONSTRAINTS (De Regels)

# Regel A: Elke staaf in de constructie MOET precies 1 balk toegewezen krijgen
for slot_id in construction_slots:
    # Zoek alle balken die überhaupt passen op dit slot
    valid_stock_for_slot = [stock_id for (stock_id, s_id) in valid_matches if s_id == slot_id]

    if not valid_stock_for_slot:
        print(f"⚠️ FATALE FOUT: Slot {slot_id} heeft GEEN ENKELE fysiek passende balk in de hele voorraad!")
    else:
        prob += pulp.lpSum([x[(stock_id, slot_id)] for stock_id in valid_stock_for_slot]) == 1

# Regel B: Reclaimed hout is uniek (Max 1x gebruiken)
for stock_id in reclaimed_items:
    valid_slots_for_stock = [s_id for (s_id_tuple, s_id) in valid_matches if s_id_tuple == stock_id]
    if valid_slots_for_stock:
        prob += pulp.lpSum([x[(stock_id, slot_id)] for slot_id in valid_slots_for_stock]) <= 1

# Regel C: Nieuw hout mag oneindig gebruikt worden (Limiet = aantal benodigde staven)
for stock_id in new_items:
    valid_slots_for_stock = [s_id for (s_id_tuple, s_id) in valid_matches if s_id_tuple == stock_id]
    if valid_slots_for_stock:
        prob += pulp.lpSum([x[(stock_id, slot_id)] for slot_id in valid_slots_for_stock]) <= len(construction_slots)

# ==========================================
# 7. OPLOSSEN EN RESULTATEN
# ==========================================
prob.solve()

print("\n" + "="*50)
print(f"STATUS OPLOSSING: {pulp.LpStatus[prob.status]}")
print("="*50)

if pulp.LpStatus[prob.status] == 'Optimal':
    total_cost = pulp.value(prob.objective)

    # Sla de winnende combinaties op in een overzichtelijke lijst
    results = []
    for j in construction_slots:
        for i in stock_items:
            match = (i, j)
            if match in x and x[match].varValue == 1:
                results.append({'Slot_ID': j, 'Assigned_Stock': i, 'CO2_Penalty': round(costs[match], 2)})

    df_results = pd.DataFrame(results)

    print(f"\n✅ Het optimale ontwerp is gevonden met een totale CO2 penalty van {total_cost:.2f} kg!")

    # ==========================================
    # 8. MERGEN MET DE ORIGINELE EDGE MATRIX
    # ==========================================
    # We gebruiken pd.merge om de nieuwe 'assigned_timber' kolom naast je bestaande V1/V2 kolommen te plakken
    # Let op: Vervang 'df_slots' door de naam van je originele edge dataframe als die anders heet (bijv. df_opt_edges)
    
    df_final_edges = pd.merge(df_slots, df_results[['edge_id', 'assigned_timber']], on='edge_id', how='left')
    
    print("\nDefinitieve Hout-Toewijzing:")
    print("-" * 50)
    print(df_results.to_string(index=False))

    # Optioneel: Exporteer dit naar CSV om in Grasshopper te kleuren
    # df_results.to_csv('definitieve_toewijzing.csv', index=False)
    
else:
    print("❌ Het algoritme kon geen oplossing vinden. Waarschijnlijk is je voorraad niet toereikend voor de opgevraagde constructie.")

# EXPORT

This is where the best parameters of the sctructure are exported, this is done in a format that can be used by grasshopper script to translate it into geometry

In [ ]:
# @title
import pandas as pd
import config
import c11_params
from geometry import generate_sample_vertices, generate_edges

# ==========================================
# 5. GEOMETRISCHE RECONSTRUCTIE VAN HET OPTIMUM
# ==========================================
print("\n📦 Exporteren van het volledige winnende ontwerp naar CSV's...")

# Haal de winnende getallen op (de 'DNA' vector)
# <-- AANGEPAST: Moet een dictionary zijn, geen getal '1'
best_parameters = {} # Vervang {} later door: optimizer.best_params
OPTIMUM_ID = 9999

def reconstruct_optimum_vertices(params, sample_id):
    """Herbouwt exact dat ene beste ontwerp."""
    # <-- AANGEPAST: params nu correct doorgegeven aan de motor
    vertices = generate_sample_vertices(sample_id, params=params)
    return pd.DataFrame(vertices)

# --- 2. Genereer de Vertices ---
df_opt_vertices = reconstruct_optimum_vertices(best_parameters, OPTIMUM_ID)

# --- 3. Genereer de Edges ---
# <-- AANGEPAST: num_samples op 1 gezet, want we bouwen maar 1 optimum!
df_opt_edges = generate_edges(num_samples=1, cells_x=c11_params.GRID_CELLS_X, cells_y=c11_params.GRID_CELLS_Y)

# BELANGRIJK: Omdat generate_edges() altijd bij sample_id=0 begint,
# forceren we de ID naar ons OPTIMUM_ID, zodat hij synchroon loopt met de vertices.
df_opt_edges['sample_id'] = OPTIMUM_ID

# --- 4. Exporteren ---
pad_vertices = config.EXPORT_PATH / 'optimum_vertices.csv'
pad_edges = config.EXPORT_PATH / 'optimum_edges.csv'

df_opt_vertices.to_csv(pad_vertices, index=False)
df_opt_edges.to_csv(pad_edges, index=False)

print(f"✅ Succes! Volledig winnende geometrie ({len(df_opt_vertices)} punten, {len(df_opt_edges)} lijnen) opgeslagen.")
print(f"   -> {pad_vertices}")
print(f"   -> {pad_edges}")